In [0]:
from pyspark.sql.functions import col,avg,count,row_number,rank,dense_rank,floor,datediff
from pyspark.sql.window import Window

df_customer=spark.table("customer")
df_order=spark.table("orders")   
df_invoice=spark.table("invoice")



In [0]:
#select highest order amount by location wise
from pyspark.sql.functions import col,max

df_customer.alias(
    "c"
).join(
      df_order.alias(
          "o"
      ),
      col("c.cust_id")==col("o.cust_id"),
      "inner"
) \
 .groupBy("c.location") \
 .agg(max(col("o.order_amount")).alias("total_purchase")
 )\
 .orderBy(col("total_purchase").desc()) \
 .show()



In [0]:
#find customers with highest total purchase in each location
from pyspark.sql.functions import col,max,sum

df_customer.alias(
    "c"
).join(
      df_order.alias(
          "o"
      ),
      col("c.cust_id")==col("o.cust_id"),
      "inner"
) \
 .groupBy("c.location","c.cust_name") \
 .agg(sum("o.order_amount").alias("total_purchase"))\
.orderBy(col("total_purchase").desc()) \
.withColumn("rn",row_number().over(
    Window.partitionBy("location").orderBy(col("total_purchase").desc())
))\
.filter(col("rn")==1)\
.show()

In [0]:

%sql
select location,total from(select c.location as location, sum(o.order_amount) as total,row_number() over(order by sum(o.order_amount)) as row_number from customer c join orders o on c.cust_id=o.cust_id group by c.location order by total desc)t where row_number=3

In [0]:
%sql
-- select location,sum(order_amount) as total from customer c join orders o on c.cust_id=o.cust_id group by location order by total;

select location,sum(order_amount) from (select location,order_amount from customer c join orders o where c.cust_id=o.cust_id) group by location;

In [0]:
#find maximum order amount
df_order.withColumn("max_order_amount",row_number().over(Window.orderBy(col("order_amount").desc()))).filter(col("max_order_amount")==1).select("order_amount").show()


In [0]:
#average time gap between order and delivery date
df_invoice.alias("i").join(
    df_order.alias("o"),col("i.order_id")==col("o.order_id"),"inner") \
        .agg(floor(avg(datediff(col("i.delivery_date"),col("o.order_date")))).alias("average gap"))\
            .show()



In [0]:
%sql
select floor(avg(datediff(i.delivery_date,o.order_date))) from orders o join invoice i on o.order_id=i.order_id

In [0]:
# identify customer details who has received their order
from pyspark.sql.functions import col
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").join(df_invoice.alias("i"),col("o.order_id")==col("i.order_id")).select("c.cust_id","c.cust_name","c.location","c.contact").filter(col("i.delivery_date").isNotNull()).show()


In [0]:
%sql
select c.cust_id as cust_id,c.cust_name as cust_name,c.location as location,c.contact as contact from 
    customer c join orders o on c.cust_id=o.cust_id
    join invoice i on o.order_id=i.order_id
    where i.delivery_date is null

In [0]:
# get customer name and their order ids
from pyspark.sql.functions import col,collect_list,sum
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").select("c.cust_name","o.order_id","o.order_amount").show()

In [0]:
%sql
select cust_name, collect_list(order_id) as order_id from (
    select c.cust_name,o.order_id from customer c join orders o on c.cust_id=o.cust_id
    ) t
     group by cust_name

In [0]:
# find total order amount and avg order amount for each customer
from pyspark.sql.functions import col
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").groupBy("cust_name").agg(sum(col("order_amount")).alias("total"),avg(col("order_amount")).alias("avg_order_amount")).show()

In [0]:
%sql
select c.cust_name,count(o.order_id),sum(o.order_amount) as total_order_amount,avg(o.order_amount) as avg_order_amount from customer c join orders o on c.cust_id=o.cust_id group by cust_name

In [0]:
# Find customers with no orders
from pyspark.sql.functions import col

df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"left_anti").select("c.cust_name").show()



In [0]:
%sql
select * from customer c left join orders o on c.cust_id=o.cust_id where o.order_id is null


In [0]:
#Find maximum order amount by location
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").groupBy("location").agg(max("order_amount").alias("max_order_amount")).orderBy(col("max_order_amount").desc()).show()

In [0]:
%sql
select c.location,max(o.order_amount) as max_order_amount from customer c join orders o on c.cust_id=o.cust_id group by c.location order by max_order_amount desc

In [0]:
# Find 3rd highest order amount
from pyspark.sql.functions import col
from pyspark.sql.window import Window
df_order.withColumn("rn",row_number().over(Window.orderBy(col("order_amount").desc()))).filter(col("rn")==3).select("order_id","order_amount").show()

In [0]:
%sql
select order_id,order_amount from (select order_id,order_amount, row_number() over(order by(order_amount) desc) as rn from orders)t where rn=3

In [0]:
#Find customer with 3rd highest total purchase
from pyspark.sql.functions import col
from pyspark.sql.window import Window
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner") \
    .groupBy("cust_name").agg(sum(col("order_amount")).alias("total"))\
    .withColumn("rn",row_number().over(Window.orderBy(col("total").desc()))) \
    .filter(col("rn")==3).show()

In [0]:
%sql
select cust_name,total,rn from (select c.cust_name,sum(o.order_amount) as total,row_number() over(order by sum(o.order_amount) desc)  as rn from customer c join orders o on c.cust_id=o.cust_id group by c.cust_name)t where rn=3

In [0]:
# Find customers who placed more than 3 orders
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner") \
    .groupBy("cust_name").agg(count("order_id").alias("total_orders")).filter(col("total_orders")>3).show()

In [0]:
%sql
select cust_name,total from (select c.cust_name,count(o.order_id) as total from customer c join orders o on c.cust_id=o.cust_id group by cust_name)t where total>3

In [0]:
#Find latest order of each customer
from pyspark.sql.functions import col
from pyspark.sql.window import Window
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner") \
    .withColumn("rn",row_number().over(Window.partitionBy("cust_name").orderBy(col("order_date").desc())))\
        .filter(col("rn")==1).select("cust_name","order_date").show()

In [0]:
%sql
select cust_name,order_date from (select c.cust_name,o.order_date,row_number() over(partition by c.cust_name 
order by o.order_date desc) as rn from customer c join orders o on c.cust_id=o.cust_id)t where rn=1; 

In [0]:
#find running total of order amount for each customer
from pyspark.sql.functions import *
from pyspark.sql.window import Window
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner") \
    .withColumn("running_total",sum(col("order_amount")).over(Window.partitionBy("cust_name").orderBy(col("order_date").asc()))).select("cust_name","order_amount","order_date","running_total").show()

In [0]:
%sql
select c.cust_name,o.order_date,o.order_amount,sum(o.order_amount) over(partition by cust_name order by order_date asc) as running_total from customer c join orders o on c.cust_id=o.cust_id

In [0]:
#find duplicate order_id for each customer
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner") \
    .groupBy("cust_name","order_id").agg(count("order_id").alias("count")).filter(col("count")>1).show()

df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner")\
    .withColumn("row_number",row_number().over(Window.partitionBy("cust_name","order_id").orderBy(col("order_id").desc()))).show() 


In [0]:
# perform masking on invoice

from pyspark.sql.functions import lit,col,min,max



# mask order_amount column
df_order = df_order.withColumn("order_amount",lit("********"))
df_order.display()
